In [1]:
import pandas as pd
import numpy as np
import os

RAW_PATH     = r"D:\Projects\End-to-end projects\17. SKU Proliferation & Rationalization\Data\Raw"
CLEANED_PATH = r"D:\Projects\End-to-end projects\17. SKU Proliferation & Rationalization\Data\Cleaned"

os.makedirs(CLEANED_PATH, exist_ok=True)

# Load all four tables
sku_master  = pd.read_csv(os.path.join(RAW_PATH, "sku_master.csv"))
sales       = pd.read_csv(os.path.join(RAW_PATH, "sales_transactions.csv"), parse_dates=["order_date"])
sku_ops     = pd.read_csv(os.path.join(RAW_PATH, "sku_ops_cost.csv"))
cfp         = pd.read_csv(os.path.join(RAW_PATH, "customer_first_purchase.csv"),
                           parse_dates=["first_order_date", "second_order_date"])

print("Raw data loaded.")
print(f"sku_master  : {sku_master.shape}")
print(f"sales       : {sales.shape}")
print(f"sku_ops     : {sku_ops.shape}")
print(f"cfp         : {cfp.shape}")

Raw data loaded.
sku_master  : (280, 9)
sales       : (85000, 9)
sku_ops     : (280, 6)
cfp         : (33881, 7)


In [2]:
tables = {
    "sku_master" : sku_master,
    "sales"      : sales,
    "sku_ops"    : sku_ops,
    "cfp"        : cfp
}

print("=== NULL CHECK ===\n")
for name, df in tables.items():
    null_counts = df.isnull().sum()
    null_counts = null_counts[null_counts > 0]
    if null_counts.empty:
        print(f"{name}: No nulls found ✓")
    else:
        print(f"\n{name} — Nulls found:")
        print(null_counts)

=== NULL CHECK ===

sku_master: No nulls found ✓
sales: No nulls found ✓
sku_ops: No nulls found ✓

cfp — Nulls found:
second_sku_id        8902
second_order_date    8902
dtype: int64


In [4]:
print("=== SKU MASTER CLEANING ===\n")
print(f"Before: {sku_master.shape[0]} rows")

# 1. Convert launch_date to datetime
sku_master["launch_date"] = pd.to_datetime(sku_master["launch_date"])

# 2. Calculate SKU age in months (from Jan 2024, start of our data window)
reference_date = pd.Timestamp("2024-01-01")
sku_master["sku_age_months"] = (
    (reference_date - sku_master["launch_date"]).dt.days / 30.44
).round(1)

# 3. Calculate gross margin % per SKU
# Why: this is the baseline profitability before complexity costs
sku_master["gross_margin_pct"] = (
    (sku_master["mrp"] - sku_master["cost_price"]) / sku_master["mrp"] * 100
).round(2)

# 4. Flag negative or zero margin SKUs (data quality issue)
bad_margin = sku_master[sku_master["gross_margin_pct"] <= 0]
print(f"SKUs with zero or negative gross margin: {len(bad_margin)}")

# 5. Remove duplicates on sku_id
dups = sku_master.duplicated(subset="sku_id").sum()
print(f"Duplicate sku_ids: {dups}")
sku_master = sku_master.drop_duplicates(subset="sku_id")

print(f"After: {sku_master.shape[0]} rows")
sku_master.head(3)

=== SKU MASTER CLEANING ===

Before: 280 rows
SKUs with zero or negative gross margin: 0
Duplicate sku_ids: 0
After: 280 rows


,sku_id,sku_name,category,subcategory,launch_date,mrp,cost_price,is_active,pack_size,sku_age_months,gross_margin_pct
0,SKU0001,Haircare Shampoo Variant 1,Haircare,Shampoo,2022-09-01,232.0,78.0,True,50ml,16.0,66.38
1,SKU0002,Haircare Shampoo Variant 2,Haircare,Shampoo,2023-05-25,967.0,277.0,True,30 tablets,7.3,71.35
2,SKU0003,Haircare Shampoo Variant 3,Haircare,Shampoo,2020-11-11,929.0,393.0,True,1kg,37.6,57.70


In [5]:
print("=== SALES TRANSACTIONS CLEANING ===\n")
print(f"Before: {sales.shape[0]} rows")

# 1. Remove duplicate order_ids
dups = sales.duplicated(subset="order_id").sum()
print(f"Duplicate order_ids: {dups}")
sales = sales.drop_duplicates(subset="order_id")

# 2. Remove returned orders from revenue analysis
# Why: returned orders inflate revenue and distort SKU performance
#      We keep return data for return rate calculation separately
returns      = sales[sales["return_flag"] == True].copy()
sales_clean  = sales[sales["return_flag"] == False].copy()
print(f"Returned orders removed : {len(returns)}")
print(f"Clean orders remaining  : {len(sales_clean)}")

# 3. Validate selling price > 0
zero_price = sales_clean[sales_clean["selling_price"] <= 0]
print(f"Zero or negative price rows: {len(zero_price)}")
sales_clean = sales_clean[sales_clean["selling_price"] > 0]

# 4. Validate units_sold > 0
zero_units = sales_clean[sales_clean["units_sold"] <= 0]
print(f"Zero or negative units rows: {len(zero_units)}")
sales_clean = sales_clean[sales_clean["units_sold"] > 0]

# 5. Only keep orders where sku_id exists in sku_master
# Why: orphan transactions (SKU deleted from master) corrupt joins
valid_skus   = sku_master["sku_id"].tolist()
orphan_rows  = sales_clean[~sales_clean["sku_id"].isin(valid_skus)]
print(f"Orphan transactions (SKU not in master): {len(orphan_rows)}")
sales_clean  = sales_clean[sales_clean["sku_id"].isin(valid_skus)]

# 6. Add revenue column
sales_clean["revenue"] = (sales_clean["units_sold"] * sales_clean["selling_price"]).round(2)

# 7. Add year_month column for trend analysis
sales_clean["year_month"] = sales_clean["order_date"].dt.to_period("M")

print(f"\nAfter cleaning: {sales_clean.shape[0]} rows")
sales_clean.head(3)

=== SALES TRANSACTIONS CLEANING ===

Before: 85000 rows
Duplicate order_ids: 0
Returned orders removed : 6809
Clean orders remaining  : 78191
Zero or negative price rows: 0
Zero or negative units rows: 0
Orphan transactions (SKU not in master): 0

After cleaning: 78191 rows


,order_id,order_date,customer_id,sku_id,channel,units_sold,selling_price,discount_pct,return_flag,revenue,year_month
0,ORD0000001,2024-03-21,CUST031397,SKU0030,D2C Website,1,276.45,0.05,False,276.45,2024-03
1,ORD0000002,2025-07-05,CUST007159,SKU0238,Amazon,1,987.05,0.05,False,987.05,2025-07
2,ORD0000003,2025-08-23,CUST014790,SKU0054,Amazon,1,1356.60,0.05,False,1356.60,2025-08


In [6]:
print("=== SKU OPS COST CLEANING ===\n")
print(f"Before: {sku_ops.shape[0]} rows")

# 1. Remove duplicates
dups = sku_ops.duplicated(subset="sku_id").sum()
print(f"Duplicate sku_ids: {dups}")
sku_ops = sku_ops.drop_duplicates(subset="sku_id")

# 2. Check for nulls
print(f"Nulls: {sku_ops.isnull().sum().sum()}")

# 3. Validate numeric ranges
print(f"\nStorage cost range   : ₹{sku_ops['monthly_storage_cost'].min()} – ₹{sku_ops['monthly_storage_cost'].max()}")
print(f"Handling cost range  : ₹{sku_ops['avg_handling_cost_per_unit'].min()} – ₹{sku_ops['avg_handling_cost_per_unit'].max()}")
print(f"Complexity score range: {sku_ops['packaging_complexity_score'].min()} – {sku_ops['packaging_complexity_score'].max()}")
print(f"Lead time range      : {sku_ops['supplier_lead_time_days'].min()} – {sku_ops['supplier_lead_time_days'].max()} days")

# 4. Calculate monthly complexity cost per SKU
# Logic: storage + (complexity score × 200 overhead proxy)
# Why: packaging complexity drives hidden costs in labelling, QC, and rework
sku_ops["monthly_complexity_cost"] = (
    sku_ops["monthly_storage_cost"] +
    (sku_ops["packaging_complexity_score"] * 200)
).round(2)

print(f"\nAfter: {sku_ops.shape[0]} rows")
sku_ops.head(3)

=== SKU OPS COST CLEANING ===

Before: 280 rows
Duplicate sku_ids: 0
Nulls: 0

Storage cost range   : ₹85.0 – ₹595.0
Handling cost range  : ₹8.31 – ₹44.94
Complexity score range: 1 – 5
Lead time range      : 7 – 45 days

After: 280 rows


,sku_id,monthly_storage_cost,min_order_quantity,avg_handling_cost_per_unit,packaging_complexity_score,supplier_lead_time_days,monthly_complexity_cost
0,SKU0001,510.0,100,9.79,2,14,910.0
1,SKU0002,336.0,100,44.44,4,14,1136.0
2,SKU0003,335.0,100,14.19,1,21,535.0


In [7]:
print("=== CUSTOMER FIRST PURCHASE CLEANING ===\n")
print(f"Before: {cfp.shape[0]} rows")

# 1. Remove duplicates
dups = cfp.duplicated(subset="customer_id").sum()
print(f"Duplicate customer_ids: {dups}")
cfp = cfp.drop_duplicates(subset="customer_id")

# 2. Nulls in second_sku_id = single-purchase customers — label them clearly
cfp["is_repeat_customer"] = cfp["second_sku_id"].notnull()
single_buyers = (~cfp["is_repeat_customer"]).sum()
repeat_buyers = cfp["is_repeat_customer"].sum()
print(f"Single-purchase customers : {single_buyers}")
print(f"Repeat customers          : {repeat_buyers}")
print(f"Repeat rate               : {repeat_buyers / len(cfp) * 100:.1f}%")

# 3. Validate first_sku_id exists in sku_master
orphan_customers = cfp[~cfp["first_sku_id"].isin(valid_skus)]
print(f"Customers with orphan first SKU: {len(orphan_customers)}")
cfp = cfp[cfp["first_sku_id"].isin(valid_skus)]

# 4. Validate ltv_90d >= 0
cfp = cfp[cfp["ltv_90d"] >= 0]

print(f"\nAfter: {cfp.shape[0]} rows")
cfp.head(3)

=== CUSTOMER FIRST PURCHASE CLEANING ===

Before: 33881 rows
Duplicate customer_ids: 0
Single-purchase customers : 8902
Repeat customers          : 24979
Repeat rate               : 73.7%
Customers with orphan first SKU: 0

After: 33881 rows


,customer_id,first_sku_id,first_order_date,second_sku_id,second_order_date,total_orders_lifetime,ltv_90d,is_repeat_customer
0,CUST000001,SKU0002,2024-08-30,SKU0017,2024-12-07,2,1740.6,True
1,CUST000002,SKU0270,2024-03-17,SKU0025,2024-09-25,6,989.1,True
2,CUST000004,SKU0271,2024-10-06,NaN,NaT,1,751.0,False


In [8]:
sku_master.to_csv(os.path.join(CLEANED_PATH, "sku_master_clean.csv"),  index=False)
sales_clean.to_csv(os.path.join(CLEANED_PATH, "sales_clean.csv"),       index=False)
sku_ops.to_csv(os.path.join(CLEANED_PATH, "sku_ops_clean.csv"),         index=False)
cfp.to_csv(os.path.join(CLEANED_PATH, "customer_first_purchase_clean.csv"), index=False)

# Also save returns separately — useful for return rate analysis later
returns.to_csv(os.path.join(CLEANED_PATH, "returns.csv"), index=False)

print("=== CLEANED FILES SAVED ===\n")
print(f"sku_master_clean            : {sku_master.shape[0]} rows")
print(f"sales_clean                 : {sales_clean.shape[0]} rows")
print(f"sku_ops_clean               : {sku_ops.shape[0]} rows")
print(f"customer_first_purchase_clean: {cfp.shape[0]} rows")
print(f"returns                     : {returns.shape[0]} rows")

=== CLEANED FILES SAVED ===

sku_master_clean            : 280 rows
sales_clean                 : 78191 rows
sku_ops_clean               : 280 rows
customer_first_purchase_clean: 33881 rows
returns                     : 6809 rows


In [9]:
print("=== CLEANING SUMMARY ===\n")

original_orders = 85000
clean_orders    = sales_clean.shape[0]
removed         = original_orders - clean_orders

print(f"Orders before cleaning  : {original_orders:,}")
print(f"Orders after cleaning   : {clean_orders:,}")
print(f"Orders removed          : {removed:,} ({removed/original_orders*100:.1f}%)")
print(f"\nRevenue (clean orders)  : ₹{sales_clean['revenue'].sum():,.0f}")
print(f"Active SKUs in sales    : {sales_clean['sku_id'].nunique()}")
print(f"Date range              : {sales_clean['order_date'].min().date()} → {sales_clean['order_date'].max().date()}")
print(f"\nRepeat customer rate    : {cfp['is_repeat_customer'].mean()*100:.1f}%")
print(f"Avg 90-day LTV          : ₹{cfp['ltv_90d'].mean():,.0f}")
print("\nAll cleaning steps complete. ✓")

=== CLEANING SUMMARY ===

Orders before cleaning  : 85,000
Orders after cleaning   : 78,191
Orders removed          : 6,809 (8.0%)

Revenue (clean orders)  : ₹91,806,635
Active SKUs in sales    : 258
Date range              : 2024-01-01 → 2025-12-31

Repeat customer rate    : 73.7%
Avg 90-day LTV          : ₹1,490

All cleaning steps complete. ✓
